In [ ]:
import os
import re
import json
import anthropic
from dotenv import load_dotenv
from openai import OpenAIError
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_deepseek import ChatDeepSeek
from langchain.prompts import PromptTemplate
from datasets import load_dataset
from langchain.llms.base import BaseLLM
from langchain.chat_models.base import BaseChatModel
from langchain.schema import HumanMessage
from langchain_core.output_parsers.json import JsonOutputParser
from langchain.pydantic_v1 import BaseModel, Field
from langchain.schema.runnable import RunnablePassthrough
from typing import List, Dict, Any
import difflib

load_dotenv()

In [ ]:
dataset = load_dataset("princeton-nlp/SWE-bench_Lite", split="test")

# Define LLMs with API keys loaded from environment variables
llms = {
    "chatgpt": ChatOpenAI(
        model_name="gpt-4o",
        openai_api_key=os.getenv("OPENAI_API_KEY")
    ),
    "claude": ChatAnthropic(
        model_name="claude-3-5-sonnet-20240620",
        anthropic_api_key=os.getenv("ANTHROPIC_API_KEY")
    ),
    "deepseek": ChatDeepSeek(
        model="deepseek-chat",
        api_key=os.getenv("DEEPSEEK_API_KEY")
    )
}

In [ ]:
# Function to add line numbers to file content
def add_line_numbers(file_content):
    lines = file_content.splitlines()
    
    # Find the maximum line length to align comments
    max_line_length = max(len(line) for line in lines) if lines else 0
    
    # Add a buffer for readability
    padding = 7  # Provides space between code and comment
    
    # Add line numbers as comments
    numbered_lines = []
    for i, line in enumerate(lines, 1):
        offset = max_line_length - len(line) + padding
        spaces = " " * offset
        numbered_lines.append(f"{line}{spaces}# Line {i}")
    
    return "\n".join(numbered_lines)

In [ ]:
# Define the expected output structure
class CodeChange(BaseModel):
    start_line: int = Field(description="The starting line number of the code to be replaced")
    end_line: int = Field(description="The ending line number of the code to be replaced")
    replace_string: str = Field(description="The new code that will replace the old code")

class CodeChanges(BaseModel):
    changes: List[CodeChange] = Field(description="List of code changes to be applied")

In [ ]:
# Prompt Template
prompt_template = PromptTemplate(
    input_variables=["problem_statement", "file_content", "file_path"],
    template="""
We are solving the following issue:
--- BEGIN ISSUE ---
{problem_statement}
--- END ISSUE ---

Below is the relevant file:
--- BEGIN FILE ---
```
{file_connumbered_file_contentent}
```
--- END FILE ---

Please fix the issues in the code. Instead of providing the complete fixed code, respond ONLY with a Git diff that shows your changes.

Output format:
1. Use the Git diff format to show the changes.
2. Start with "--- a/{file_path}" for the original code.
3. Start with "+++ b/{file_path}" for the modified code.
4. Use @@ to indicate the line numbers and context.
5. Use - to show removed lines and + to show added lines.
6. Ensure the output is a valid Git diff format so I can easily extract it.

Your response should ONLY contain the Git diff and nothing else.
"""
)

In [ ]:
# # Create output parser
# parser = JsonOutputParser(pydantic_object=CodeChanges)

# # Prompt Template with clear instructions for JSON output
# prompt_template = PromptTemplate(
#     input_variables=["problem_statement", "numbered_file_content", "file_path"],
#     template="""
# We are solving the following issue:
# --- BEGIN ISSUE ---
# {problem_statement}
# --- END ISSUE ---

# Below is the relevant file with line numbers:
# --- BEGIN FILE ---
# ```
# {numbered_file_content}
# ```
# --- END FILE ---

# Please fix the issues in the code. Instead of providing the complete fixed code, respond with a JSON object that specifies exactly which parts of the code need to be changed.

# For each change you want to make, specify:
# 1. The start line number
# 2. The end line number
# 3. The exact replacement string (without line number comments)

# Follow these guidelines:
# - Please carefully ensure the accuracy of indentation and space numbers in front of each line.
# - Make minimal changes necessary to fix the issue
# - Do not modify comments unless they're incorrect
# - Ignore line numbers in your replacement string
# - Your changes should be non-overlapping

# Output format:
# ```json
# {{
#   "changes": [
#     {{
#       "start_line": <int>,
#       "end_line": <int>,
#       "replace_string": <string>
#     }},
#     ...more changes if needed
#   ]
# }}
# ```

# Ensure your response only contains valid JSON that conforms to this schema.
# """
# )


In [ ]:
def extract_modified_file_path(patch):
    """Extracts the modified file path from the first line of a Git diff."""
    match = re.search(r'diff --git a/(.*?) b/', patch)
    return match.group(1) if match else None

def apply_changes(original_content, changes):
    """Apply the changes to the original content."""
    lines = original_content.splitlines()
    
    # Sort changes in reverse order of start_line to avoid index shifting
    sorted_changes = sorted(changes, key=lambda x: x["start_line"], reverse=True)
    
    for change in sorted_changes:
        start_idx = change["start_line"] - 1  # Convert to 0-indexed
        end_idx = change["end_line"] - 1      # Convert to 0-indexed
        replace_lines = change["replace_string"].splitlines()
        
        # Replace the specified lines
        lines[start_idx:end_idx + 1] = replace_lines
    
    return "\n".join(lines)

def generate_diff(original_content, modified_content, file_path):
    """Generate a unified diff format."""
    original_lines = original_content.splitlines(keepends=True)
    modified_lines = modified_content.splitlines(keepends=True)
    
    diff = difflib.unified_diff(
        original_lines,
        modified_lines,
        fromfile=f"a/{file_path}",
        tofile=f"b/{file_path}",
        n=3  # Context lines
    )
    
    return "".join(diff)

In [ ]:
# def process_task(task, llm_name):
#     """Processes a single task using the specified LLM with the new approach."""
#     try:
#         instance_id = task["instance_id"]
#         problem_statement = task["problem_statement"]
#         patch = task["patch"]
        
#         # Extract file path from patch
#         file_path = extract_modified_file_path(patch)
#         if not file_path:
#             print(f"[Warning] No file path found for {instance_id}")
#             return
        
#         # Read file content
#         file_full_path = f"./codebases/{instance_id}/{file_path}"
#         if not os.path.exists(file_full_path):
#             print(f"[Error] File not found: {file_full_path}")
#             return
        
#         with open(file_full_path, "r", encoding="utf-8") as f:
#             original_file_content = f.read()
        
#         # Add line numbers to file content
#         numbered_file_content = add_line_numbers(original_file_content)
        
#         # Set up the chain with the output parser
#         llm = llms[llm_name]
#         chain = (
#             {"problem_statement": RunnablePassthrough(), 
#              "numbered_file_content": RunnablePassthrough(), 
#              "file_path": RunnablePassthrough()}
#             | prompt_template
#             | llm
#             | parser
#         )
        
#         # Invoke the chain
#         response = chain.invoke({
#             "problem_statement": problem_statement,
#             "numbered_file_content": numbered_file_content,
#             "file_path": file_path
#         })
        
#         # Apply the changes to get the fixed file content
#         fixed_file_content = apply_changes(original_file_content, response["changes"])
        
#         # Generate the unified diff
#         diff_output = generate_diff(original_file_content, fixed_file_content, file_path)
        
#         # Save both the JSON response and the diff
#         os.makedirs(f"./test_outputs/{llm_name}/json_format", exist_ok=True)
#         with open(f"./test_outputs/{llm_name}/json_format/{instance_id}.json", "w", encoding="utf-8") as f:
#             json.dump(response, f, indent=2)
        
#         output_path = f"./test_outputs/{llm_name}/json_format/{instance_id}.diff"
#         with open(output_path, "w", encoding="utf-8") as f:
#             f.write(diff_output)
        
#         print(f"[Success] Diff saved: {output_path}")
#     except Exception as e:
#         print(f"[Error] Failed to process {instance_id} with {llm_name}: {str(e)}")

In [ ]:
# Iterate through dataset and process each task
llm_name = 'chatgpt'
# for index, task in enumerate(dataset):
# process_task(task, llm_name)
# process_task(dataset[0], llm_name)


In [ ]:
task = dataset[3]


try:
    instance_id = task["instance_id"]
    problem_statement = task["problem_statement"]
    patch = task["patch"]
    
    # Extract file path from patch
    file_path = extract_modified_file_path(patch)
    if not file_path:
        print(f"[Warning] No file path found for {instance_id}")
        raise
    
    # Read file content
    file_full_path = f"./codebases/{instance_id}/{file_path}"
    if not os.path.exists(file_full_path):
        print(f"[Error] File not found: {file_full_path}")
        raise
    
    with open(file_full_path, "r", encoding="utf-8") as f:
        original_file_content = f.read()
    
    # Add line numbers to file content
    numbered_file_content = add_line_numbers(original_file_content)
    
    # Set up the chain with the output parser
    llm = llms[llm_name]
    chain = (
        {"problem_statement": RunnablePassthrough(), 
            "numbered_file_content": RunnablePassthrough(), 
            "file_path": RunnablePassthrough()}
        | prompt_template
        | llm
        | parser
    )
    
    # Invoke the chain
    response = chain.invoke({
        "problem_statement": problem_statement,
        "numbered_file_content": numbered_file_content,
        "file_path": file_path
    })
    
    # Apply the changes to get the fixed file content
    fixed_file_content = apply_changes(original_file_content, response["changes"])
    
    # Generate the unified diff
    diff_output = generate_diff(original_file_content, fixed_file_content, file_path)
    
    # Save both the JSON response and the diff
    os.makedirs(f"./test_outputs/{llm_name}/json_format", exist_ok=True)
    with open(f"./test_outputs/{llm_name}/json_format/{instance_id}.json", "w", encoding="utf-8") as f:
        json.dump(response, f, indent=2)
    
    output_path = f"./test_outputs/{llm_name}/json_format/{instance_id}.diff"
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(diff_output)
    
    print(f"[Success] Diff saved: {output_path}")
except Exception as e:
    print(f"[Error] Failed to process {instance_id} with {llm_name}: {str(e)}")

In [ ]:
print(response)

In [ ]:
print(numbered_file_content)